# Finetune MAE Trajectories for Polymer Tasks

This notebook summarizes the finetuning behaviour of the dynamic pretrain suite.
For each PI property, we track the MAE reported after finetuning on models
trained with progressively larger pretrain task sequences. The goal is to highlight
how performance evolves as the target task is introduced into the pretrain stages.

In [1]:
import json
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

In [2]:
ARTIFACT_ROOT = Path("../artifacts/radonpy_polyinfo_pretrain_finetune_runs_251206")
RUN_PATTERN = "run*"
# FINETUNE_TASKS = [
#     "density",
#     "Rg",
#     "r2",
#     "self-diffusion",
#     "Cp",
#     "Cv",
#     "linear_expansion",
#     "refractive_index",
#     "tg",
# ]

FINETUNE_TASKS = [
    'Tg',
    'Thermal decomposition temp-Temperature',
    'Tm',
    'Density',
    'Tensile stress at break',
    'Elongation at break',
    'Gas permeability coefficient O2',
    'Water absorption',
    'Dielectric const AC freq-all',
    'Tensile modulus',
    'Solubility parameter',
    'Gas permeability coefficient CO2',
    'Heat of fusion',
    'Gas permeability coefficient N2',
    'Refractive index',
    'Linear expansion coefficient',
    'Dielectric const AC freq-low',
    'Volume expansion coefficient',
    'Tensile storage modulus freq-all',
    'Crystallization temp',
    'Tensile stress at yield',
    'Contact angle',
    'Surface tension',
    'Tensile storage modulus loss tangent freq-all',
    'Cp',
    'Tensile storage modulus freq-middle',
    'Tensile loss modulus freq-all',
    'Dielectric const AC freq-high',
    'Dielectric const AC freq-middle',
    'Vicat softening temp',
    'Brittleness temp',
    'Water vapor transmission',
    'Hansen parameter delta-d',
    'Hansen parameter delta-p',
    'Heat of crystallization',
    'Hansen parameter delta-h',
    'Shear storage modulus freq-all',
    'Elongation at yield',
    'Tensile storage modulus freq-high',
    'Tensile storage modulus loss tangent freq-high',
    'Shear storage modulus loss tangent freq-all',
    'Tensile storage modulus loss tangent freq-middle',
    'HDT',
    'Shear storage modulus freq-low',
    'Tensile storage modulus freq-low',
    'Tensile loss modulus freq-high',
    'Thermal conductivity',
    'Shear storage modulus loss tangent freq-low',
    'Stress-optical coefficient',
    'Dielectric const DC',
    'Flexural modulus',
    'Flexural storage modulus freq-all',
    'Tensile loss modulus freq-middle',
    'Shore hardness',
    'Flexural storage modulus freq-low',
    'Growth rate of crystal',
    'Tensile storage modulus loss tangent freq-low',
    'Flexural stress at break',
    'Tensile loss modulus freq-low',
    'Charpy impact',
    'Compressive modulus',
]

if not ARTIFACT_ROOT.exists():
    raise FileNotFoundError(f"Expected artifact directory at {ARTIFACT_ROOT}")

In [3]:
def load_finetune_records(artifact_root: Path) -> pd.DataFrame:
    records: list[dict[str, object]] = []
    for run_dir in sorted(artifact_root.glob(RUN_PATTERN)):
        if not run_dir.is_dir():
            continue
        run_label = run_dir.name
        for stage_dir in sorted(run_dir.glob("pretrain_stage*")):
            pretrain_metrics_path = stage_dir / "prediction" / "metrics.json"
            if not pretrain_metrics_path.exists():
                continue
            with pretrain_metrics_path.open("r", encoding="utf-8") as handle:
                pretrain_payload = json.load(handle)
            stage_sequence = pretrain_payload.get("task_sequence") or []
            try:
                stage_index = int(stage_dir.name.split("pretrain_stage")[1].split("_")[0])
            except (IndexError, ValueError):
                continue
            finetune_root = stage_dir / "finetune"
            if not finetune_root.exists():
                continue
            for finetune_stage in sorted(finetune_root.iterdir()):
                metrics_path = finetune_stage / "prediction" / "metrics.json"
                if not metrics_path.exists():
                    continue
                with metrics_path.open("r", encoding="utf-8") as handle:
                    payload = json.load(handle)
                metrics_block = payload.get("metrics", {})
                if not metrics_block:
                    continue
                # Each finetune metrics.json contains a single task entry
                task_name, metric_values = next(iter(metrics_block.items()))
                mae = metric_values.get("mae")
                if mae is None:
                    continue
                records.append(
                    {
                        "run": run_label,
                        "stage": stage_index,
                        "property": task_name,
                        "mae": float(mae),
                        "task_sequence": stage_sequence,
                        "property_in_pretrain": task_name in stage_sequence,
                    }
                )
    frame = pd.DataFrame.from_records(records)
    if frame.empty:
        raise RuntimeError("No finetune metrics found in the artifact tree.")
    frame.sort_values(["property", "run", "stage"], inplace=True)
    frame.reset_index(drop=True, inplace=True)
    return frame


finetune_df = load_finetune_records(ARTIFACT_ROOT)
finetune_df.head()

,run,stage,property,mae,task_sequence,property_in_pretrain
0,run01,1,Brittleness temp,32.253574,[TC_dihed],False
1,run01,2,Brittleness temp,30.564400,"[TC_dihed, sp_ele_long]",False
2,run01,3,Brittleness temp,31.926353,"[TC_dihed, sp_ele_long, dielectric_const_dc]",False
3,run01,4,Brittleness temp,34.134763,"[TC_dihed, sp_ele_long, dielectric_const_dc, T...",False
4,run01,5,Brittleness temp,34.771431,"[TC_dihed, sp_ele_long, dielectric_const_dc, T...",False


In [6]:
finetune_df

,run,stage,property,mae,task_sequence,property_in_pretrain
0,run01,1,Brittleness temp,32.253574,[TC_dihed],False
1,run01,2,Brittleness temp,30.564400,"[TC_dihed, sp_ele_long]",False
2,run01,3,Brittleness temp,31.926353,"[TC_dihed, sp_ele_long, dielectric_const_dc]",False
3,run01,4,Brittleness temp,34.134763,"[TC_dihed, sp_ele_long, dielectric_const_dc, T...",False
4,run01,5,Brittleness temp,34.771431,"[TC_dihed, sp_ele_long, dielectric_const_dc, T...",False
...,...,...,...,...,...,...
33890,run13,36,Water vapor transmission,NaN,"[sp_ele_long, qm_dipole_monomer1, self-diffusi...",False
33891,run13,37,Water vapor transmission,NaN,"[sp_ele_long, qm_dipole_monomer1, self-diffusi...",False
33892,run13,38,Water vapor transmission,NaN,"[sp_ele_long, qm_dipole_monomer1, self-diffusi...",False
33893,run13,39,Water vapor transmission,NaN,"[sp_ele_long, qm_dipole_monomer1, self-diffusi...",False


In [4]:
sns.set_theme(style="whitegrid", context="talk")
properties = [prop for prop in FINETUNE_TASKS if prop in finetune_df["property"].unique()]
runs = sorted(finetune_df["run"].unique())
palette = dict(zip(runs, sns.color_palette("tab10", n_colors=len(runs))))

In [ ]:
max_stage = finetune_df["stage"].max()

for idx, property_name in enumerate(properties):
    subset = finetune_df[finetune_df["property"] == property_name]
    if subset.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
    trend_points: list[pd.DataFrame] = []

    for run_label, run_slice in subset.groupby("run"):
        run_slice = run_slice.sort_values("stage")
        ax.plot(
            run_slice["stage"],
            run_slice["mae"],
            marker="o",
            linewidth=2.0,
            color=palette[run_label],
            alpha=0.55,
            label=f"_run_{run_label}",
        )
        post_slice = run_slice[run_slice["property_in_pretrain"]]
        if not post_slice.empty:
            ax.plot(
                post_slice["stage"],
                post_slice["mae"],
                marker="o",
                linewidth=3.6,
                color=palette[run_label],
                alpha=0.95,
                label=f"_post_{run_label}",
            )
            inclusion_stage = post_slice["stage"].iloc[0]
            inclusion_mae = post_slice["mae"].iloc[0]
            ax.scatter(
                [inclusion_stage],
                [inclusion_mae],
                color=palette[run_label],
                edgecolor="black",
                linewidths=2.0,
                s=180,
                zorder=6,
            )
        trend_points.append(run_slice[["stage", "mae"]])

    if trend_points:
        trend_df = pd.concat(trend_points, ignore_index=True)
        trend_df = trend_df.groupby("stage", as_index=False)["mae"].mean()
        if len(trend_df) >= 2:
            coeffs = np.polyfit(trend_df["stage"], trend_df["mae"], deg=1)
            trend_poly = np.poly1d(coeffs)
            stage_grid = np.linspace(trend_df["stage"].min(), trend_df["stage"].max(), 200)
            ax.plot(
                stage_grid,
                trend_poly(stage_grid),
                color="black",
                linewidth=2.5,
                linestyle="--",
                label="_trend",
            )

    pretty_name = property_name.replace("_", " ").title()
    ax.set_title(f"{pretty_name} (Fine-tuning)", fontsize=20, pad=18)

    trend_handle = Line2D([], [], color="black", linewidth=2.5, linestyle="--", label="Linear trend")
    pretrain_handle = Line2D(
        [],
        [],
        marker="o",
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=2.0,
        markersize=12,
        label="Property added to pretrain",
    )

    legend = ax.legend(
        [trend_handle, pretrain_handle],
        ["Linear trend", "Finetune property\nadded to pretrain"],
        frameon=True,
        fontsize=11,
        loc="best",
        fancybox=True,
        shadow=True,
    )
    legend.get_frame().set_alpha(0.9)
    legend.get_frame().set_edgecolor("#333333")
    legend.get_frame().set_linewidth(1.2)

    ax.set_xlim(0.5, max_stage + 0.5)
    ax.set_xlabel("Pretrain Stage", fontsize=14)
    ax.set_ylabel("MAE", fontsize=14)
    ax.tick_params(axis="both", labelsize=12)
    ax.grid(True, which="major", linewidth=0.6, alpha=0.4)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)
        spine.set_color("black")

    for line in ax.lines:
        line.set_solid_capstyle("round")

    plt.tight_layout()

Each panel isolates a single property. Semi-transparent lines trace MAE progression per run, bold segments emphasise epochs after the property joins the pretrain curriculum, and a dashed linear trendline summarises the overall trajectory.